# CYP2D6 Genotype-only CNN — Diplotype-disjoint 10-fold CV

Main-performance evaluation using all `dataset.csv`. Exact diplotypes are group-held-out with `StratifiedGroupKFold`. `A; B` and `B; A` are treated as the same group. Each outer-test diplotype is unseen during both training and early-stopping validation.

## 1. Setup

In [1]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, log_loss)
import matplotlib.pyplot as plt

SEED = 42
N_SPLITS = 5
INNER_SPLITS = 5
BATCH_SIZE = 32

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything()

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


/data/home/bmi-lab/anaconda3/envs/nayb37/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:1


## 2. Load all dataset and define diplotype groups

In [3]:
# data_576.csv 전체를 사용합니다.
DATA_CSV = 'dataset.csv'

data = pd.read_csv(DATA_CSV).copy()
required = {'Variant', 'Class'}
assert required.issubset(data.columns), f'Missing columns: {required - set(data.columns)}'

def canonical_diplotype(v):
    # A; B와 B; A를 같은 diplotype group으로 처리
    alleles = [x.strip() for x in str(v).split(';') if x.strip()]
    return '; '.join(sorted(alleles))

data['DiplotypeGroup'] = data['Variant'].map(canonical_diplotype)
label_encoder = LabelEncoder()
data['ClassEncoded'] = label_encoder.fit_transform(data['Class'])
class_mapping = {i: c for i, c in enumerate(label_encoder.classes_)}
print('Shape:', data.shape)
print('Unique diplotype groups:', data['DiplotypeGroup'].nunique())
print('Unique drugs:', data['Drug'].nunique() if 'Drug' in data else 'NA')
print('Class mapping:', class_mapping)
print(data['Class'].value_counts())


Shape: (622, 12)
Unique diplotype groups: 104
Unique drugs: 59
Class mapping: {0: 'de', 1: 'f', 2: 'nor'}
de     250
f      195
nor    177
Name: Class, dtype: int64


## 3. Load CYP2D6 one-hot matrices

In [4]:
ONEHOT_FOLDER = 'onehot_7'
onehot_files = os.listdir(ONEHOT_FOLDER)
onehot_matrices = {}
max_rows = 0
for file in onehot_files:
    if not file.endswith('.csv'): continue
    variant_name = file.split('.csv')[0].replace('_', '.')
    matrix = pd.read_csv(os.path.join(ONEHOT_FOLDER, file), header=None).values
    onehot_matrices[variant_name] = matrix
    max_rows = max(max_rows, matrix.shape[0])
print('Loaded matrices:', len(onehot_matrices), '| max rows:', max_rows)

def pad_matrix(matrix, target_rows=4330, target_cols=7):
    current_rows, current_cols = matrix.shape
    padded = torch.zeros((target_rows, target_cols), dtype=torch.float32)
    padded[:current_rows, :current_cols] = torch.tensor(matrix, dtype=torch.float32)
    return padded
for key in list(onehot_matrices):
    onehot_matrices[key] = pad_matrix(onehot_matrices[key])

def map_variant_to_onehot(variant):
    variants = [x.strip() for x in str(variant).split(';')]
    matrices = []
    for v in variants[:2]:
        if v not in onehot_matrices:
            raise KeyError(f'One-hot matrix not found: {v}')
        matrices.append(onehot_matrices[v])
    while len(matrices) < 2:
        matrices.append(torch.zeros((4330, 7), dtype=torch.float32))
    return torch.vstack(matrices[:2])

X = torch.stack([map_variant_to_onehot(v) for v in data['Variant']])
y = torch.tensor(data['ClassEncoded'].values, dtype=torch.long)
groups = data['DiplotypeGroup'].values
print('X:', X.shape, 'y:', y.shape)


Loaded matrices: 361 | max rows: 4330
X: torch.Size([622, 8660, 7]) y: torch.Size([622])


## 4. Create the shared 10-fold diplotype-disjoint outer split

In [5]:
# 동일한 outer split을 두 모델에서 재현할 수 있도록 fold assignment를 생성/저장합니다.
outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_id = np.full(len(data), -1, dtype=int)

for fold, (outer_train_idx, test_idx) in enumerate(outer_cv.split(np.zeros(len(y)), y.numpy(), groups)):
    train_groups = set(groups[outer_train_idx]); test_groups = set(groups[test_idx])
    overlap = train_groups & test_groups
    assert len(overlap) == 0, f'Fold {fold+1}: diplotype leakage detected: {overlap}'
    fold_id[test_idx] = fold + 1
    counts = np.bincount(y.numpy()[test_idx], minlength=len(label_encoder.classes_))
    print(f'Fold {fold+1}: train={len(outer_train_idx)}, test={len(test_idx)}, '
          f'train groups={len(train_groups)}, test groups={len(test_groups)}, overlap=0, class counts={counts.tolist()}')

assert np.all(fold_id > 0)
split_df = data[['Variant','DiplotypeGroup','Drug','Class']].copy()
split_df['fold'] = fold_id
split_df.to_csv('diplotype_grouped_5fold_assignments.csv', index=False)
print('Saved: diplotype_grouped_5fold_assignments.csv')


Fold 1: train=530, test=92, train groups=84, test groups=20, overlap=0, class counts=[38, 31, 23]
Fold 2: train=542, test=80, train groups=88, test groups=16, overlap=0, class counts=[59, 15, 6]
Fold 3: train=392, test=230, train groups=80, test groups=24, overlap=0, class counts=[65, 80, 85]
Fold 4: train=490, test=132, train groups=81, test groups=23, overlap=0, class counts=[37, 45, 50]
Fold 5: train=534, test=88, train groups=83, test groups=21, overlap=0, class counts=[51, 24, 13]
Saved: diplotype_grouped_5fold_assignments.csv


## 5. Metrics and inner validation split

In [6]:
def compute_metrics(y_true, probs, n_classes=3):
    y_true = np.asarray(y_true); probs = np.asarray(probs)
    pred = probs.argmax(axis=1)
    out = {
        'accuracy': accuracy_score(y_true, pred),
        'precision_macro': precision_score(y_true, pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, pred, average='weighted', zero_division=0),
        'nll': log_loss(y_true, probs, labels=list(range(n_classes)))
    }
    try: out['auroc_macro_ovr'] = roc_auc_score(y_true, probs, multi_class='ovr', average='macro')
    except ValueError: out['auroc_macro_ovr'] = np.nan
    aps=[]
    for c in range(n_classes):
        binary=(y_true==c).astype(int)
        aps.append(average_precision_score(binary, probs[:,c]) if binary.sum()>0 else np.nan)
    out['aupr_macro_ovr'] = float(np.nanmean(aps))
    return out

def make_inner_split(outer_train_idx, fold):
    # outer-test와 완전히 분리된 validation set. inner split도 diplotype-disjoint.
    y_outer = y.numpy()[outer_train_idx]
    g_outer = groups[outer_train_idx]
    inner_cv = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED + fold)
    inner_train_rel, val_rel = next(inner_cv.split(np.zeros(len(outer_train_idx)), y_outer, g_outer))
    train_idx = outer_train_idx[inner_train_rel]
    val_idx = outer_train_idx[val_rel]
    assert not (set(groups[train_idx]) & set(groups[val_idx]))
    assert not (set(groups[train_idx]) & set(groups[fold_id == fold]))
    return train_idx, val_idx


## 6. Dataset and original genotype-only CNN

In [7]:
class GenotypeDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx].float().unsqueeze(0), self.y[idx].long()

dataset = GenotypeDataset(X, y)

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=(20, 7), stride=(1, 1))
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 1))  # 추가된 풀링 층
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=(20, 1), stride=(1, 1))
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 1))  # 추가된 풀링 층
        
        # 풀링 층 추가로 인한 출력 크기 변경을 고려한 새로운 계산
        self.fc1 = nn.Linear(16 * ((((8660 - 20) // 2) - 20) // 2), 128)
        self.fc2 = nn.Linear(128, 3)  # 출력 층

    def forward(self, x):
        x = self.conv1(x)
        x = nn.ReLU()(x)
        x = self.pool1(x)  # 풀링 적용
        x = self.conv2(x)
        x = nn.ReLU()(x)
        x = self.pool2(x)  # 풀링 적용
        x = x.view(x.size(0), -1)  # 평탄화
        x = self.fc1(x)
        x = nn.ReLU()(x)
        x = self.fc2(x)
        return x

## Model training and outer-fold evaluation

In [8]:
# ==========================================
# Model training and outer-fold evaluation
# ==========================================
EPOCHS = 100
PATIENCE = 30
LR = 0.0005

criterion = nn.CrossEntropyLoss()

fold_rows = []
oof_probs = np.zeros((len(data), 3), dtype=float)
oof_pred = np.full(len(data), -1)

for fold in range(1, N_SPLITS + 1):
    print('\n' + '=' * 90)
    print(f'OUTER FOLD {fold}/{N_SPLITS}')
    print('=' * 90)

    test_idx = np.where(fold_id == fold)[0]
    outer_train_idx = np.where(fold_id != fold)[0]

    train_idx, val_idx = make_inner_split(
        outer_train_idx, fold
    )

    # ----------------------------------------------------------
    # Check diplotype leakage
    # ----------------------------------------------------------
    train_groups = set(groups[train_idx])
    val_groups = set(groups[val_idx])
    test_groups = set(groups[test_idx])

    assert train_groups.isdisjoint(val_groups)
    assert train_groups.isdisjoint(test_groups)
    assert val_groups.isdisjoint(test_groups)

    print(
        f'Train      : {len(train_idx)} samples / '
        f'{len(train_groups)} diplotypes'
    )
    print(
        f'Validation : {len(val_idx)} samples / '
        f'{len(val_groups)} diplotypes'
    )
    print(
        f'Outer Test : {len(test_idx)} samples / '
        f'{len(test_groups)} diplotypes'
    )

    print(
        'Train class counts:',
        data.iloc[train_idx]['Class'].value_counts().to_dict()
    )
    print(
        'Val class counts  :',
        data.iloc[val_idx]['Class'].value_counts().to_dict()
    )
    print(
        'Test class counts :',
        data.iloc[test_idx]['Class'].value_counts().to_dict()
    )

    # ----------------------------------------------------------
    # DataLoader
    # ----------------------------------------------------------
    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        Subset(dataset, test_idx),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # ----------------------------------------------------------
    # Model initialization
    # ----------------------------------------------------------
    seed_everything(SEED + fold)

    model = SimpleCNN().to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    # Best model is selected using validation Macro-F1
    best_val_f1 = -np.inf
    best_epoch = None
    patience_counter = 0

    best_path = (
        f'CNN_diplotype_grouped_fold_{fold}.pth'
    )

    history = []

    # ==========================================================
    # Training
    # ==========================================================
    for epoch in range(EPOCHS):

        # ------------------------------------------------------
        # Train
        # ------------------------------------------------------
        model.train()

        total_train_loss = 0.0
        total_train = 0
        train_true = []
        train_pred = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_train_loss += (
                loss.item() * yb.size(0)
            )
            total_train += yb.size(0)

            pred = logits.argmax(dim=1)

            train_true.extend(
                yb.detach().cpu().numpy()
            )
            train_pred.extend(
                pred.detach().cpu().numpy()
            )

        train_loss = (
            total_train_loss / total_train
        )

        train_true = np.array(train_true)
        train_pred = np.array(train_pred)

        train_acc = accuracy_score(
            train_true,
            train_pred
        )

        # ------------------------------------------------------
        # Validation
        # ------------------------------------------------------
        model.eval()

        total_val_loss = 0.0
        total_val = 0

        val_true = []
        val_probs = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                logits = model(xb)
                loss = criterion(logits, yb)

                total_val_loss += (
                    loss.item() * yb.size(0)
                )
                total_val += yb.size(0)

                probs = F.softmax(
                    logits,
                    dim=1
                )

                val_true.append(
                    yb.cpu().numpy()
                )
                val_probs.append(
                    probs.cpu().numpy()
                )

        val_loss = (
            total_val_loss / total_val
        )

        val_true = np.concatenate(
            val_true
        )

        val_probs = np.vstack(
            val_probs
        )

        val_pred = val_probs.argmax(
            axis=1
        )

        val_acc = accuracy_score(
            val_true,
            val_pred
        )

        val_precision = precision_score(
            val_true,
            val_pred,
            average='macro',
            zero_division=0
        )

        val_recall = recall_score(
            val_true,
            val_pred,
            average='macro',
            zero_division=0
        )

        val_f1 = f1_score(
            val_true,
            val_pred,
            average='macro',
            zero_division=0
        )

        # ------------------------------------------------------
        # Save training history
        # ------------------------------------------------------
        history.append({
            'fold': fold,
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_accuracy': train_acc,
            'val_loss': val_loss,
            'val_accuracy': val_acc,
            'val_precision_macro': val_precision,
            'val_recall_macro': val_recall,
            'val_f1_macro': val_f1
        })

        # ------------------------------------------------------
        # Print epoch result
        # ------------------------------------------------------
        print(
            f'Epoch [{epoch + 1:02d}/{EPOCHS}] | '
            f'Train Loss {train_loss:.4f} '
            f'Acc {train_acc:.4f} | '
            f'Val Loss {val_loss:.4f} '
            f'Acc {val_acc:.4f} | '
            f'Val Macro-Precision {val_precision:.4f} '
            f'Macro-Recall {val_recall:.4f} '
            f'Macro-F1 {val_f1:.4f}'
        )

        # ------------------------------------------------------
        # Early stopping:
        # Best checkpoint by Validation Macro-F1
        # ------------------------------------------------------
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch + 1
            patience_counter = 0

            torch.save(
                model.state_dict(),
                best_path
            )

        else:
            patience_counter += 1

            if patience_counter >= PATIENCE:
                print('Early stopping applied.')
                break

    # ----------------------------------------------------------
    # Save training history
    # ----------------------------------------------------------
    pd.DataFrame(history).to_csv(
        f'genotype_only_training_history_fold_{fold}.csv',
        index=False
    )

    # ==========================================================
    # Reload BEST checkpoint
    # ==========================================================
    model.load_state_dict(
        torch.load(
            best_path,
            map_location=device
        )
    )

    model.eval()

    # ----------------------------------------------------------
    # Validation performance of BEST model
    # ----------------------------------------------------------
    best_val_true = []
    best_val_probs = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)

            logits = model(xb)

            probs = F.softmax(
                logits,
                dim=1
            )

            best_val_true.append(
                yb.numpy()
            )
            best_val_probs.append(
                probs.cpu().numpy()
            )

    best_val_true = np.concatenate(
        best_val_true
    )

    best_val_probs = np.vstack(
        best_val_probs
    )

    best_val_metrics = compute_metrics(
        best_val_true,
        best_val_probs
    )

    # ----------------------------------------------------------
    # Outer test evaluation
    # ----------------------------------------------------------
    test_true = []
    test_probs = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)

            logits = model(xb)

            probs = F.softmax(
                logits,
                dim=1
            )

            test_true.append(
                yb.numpy()
            )
            test_probs.append(
                probs.cpu().numpy()
            )

    yt = np.concatenate(
        test_true
    )

    pr = np.vstack(
        test_probs
    )

    test_metrics = compute_metrics(
        yt,
        pr
    )

    # ----------------------------------------------------------
    # Fold result
    # ----------------------------------------------------------
    print(f'Best epoch: {best_epoch}')
    print(
        f'Best Val Macro-F1: '
        f'{best_val_metrics["f1_macro"]:.4f}'
    )

    print(
        '*** OUTER TEST — UNSEEN DIPLOTYPES ***'
    )

    print(
        f'Acc {test_metrics["accuracy"]:.4f} | '
        f'Macro-Precision '
        f'{test_metrics["precision_macro"]:.4f} | '
        f'Macro-Recall '
        f'{test_metrics["recall_macro"]:.4f} | '
        f'Macro-F1 '
        f'{test_metrics["f1_macro"]:.4f} | '
        f'Macro-AUROC '
        f'{test_metrics["auroc_macro_ovr"]:.4f} | '
        f'Macro-AUPR '
        f'{test_metrics["aupr_macro_ovr"]:.4f}'
    )

    # ----------------------------------------------------------
    # Store fold metrics
    # ----------------------------------------------------------
    fold_row = {
        'fold': fold,
        'best_epoch': best_epoch,

        'n_train': len(train_idx),
        'n_val': len(val_idx),
        'n_test': len(test_idx),

        'n_train_groups': len(train_groups),
        'n_val_groups': len(val_groups),
        'n_test_groups': len(test_groups),

        'best_val_f1': (
            best_val_metrics['f1_macro']
        ),

        **test_metrics
    }

    fold_rows.append(
        fold_row
    )

    # ----------------------------------------------------------
    # Store OOF predictions
    # ----------------------------------------------------------
    oof_probs[test_idx] = pr
    oof_pred[test_idx] = pr.argmax(
        axis=1
    )

    del model, optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ==============================================================
# Fold-level results
# ==============================================================
fold_results = pd.DataFrame(
    fold_rows
).sort_values('fold')

fold_results.to_csv(
    'genotype_only_diplotype_grouped_5fold_metrics.csv',
    index=False
)

print('\n' + '=' * 90)
print('FOLD RESULTS')
print('=' * 90)

display(fold_results)


# ==============================================================
# Mean ± SD across outer folds
# ==============================================================
metric_cols = [
    'accuracy',
    'precision_macro',
    'recall_macro',
    'f1_macro',
    'f1_weighted',
    'auroc_macro_ovr',
    'aupr_macro_ovr',
    'nll'
]

summary = pd.DataFrame({
    'mean': fold_results[
        metric_cols
    ].mean(),

    'sd': fold_results[
        metric_cols
    ].std(ddof=1)
})

print('\n' + '=' * 90)
print('MEAN ± SD ACROSS OUTER FOLDS')
print('=' * 90)

display(summary)


# ==============================================================
# Pooled OOF performance
# ==============================================================
pooled = compute_metrics(
    y.numpy(),
    oof_probs
)

print('\n' + '=' * 90)
print('POOLED OUT-OF-FOLD PERFORMANCE')
print('=' * 90)

print(pooled)


# ==============================================================
# Save OOF predictions
# ==============================================================
oof = data.copy()

oof['fold'] = fold_id
oof['y_true'] = y.numpy()
oof['y_pred'] = oof_pred

for c, name in class_mapping.items():
    oof[f'prob_{name}'] = (
        oof_probs[:, c]
    )

oof.to_csv(
    'genotype_only_diplotype_grouped_5fold_oof_predictions.csv',
    index=False
)

print(
    '\nSaved fold metrics, '
    'training histories, and OOF predictions.'
)


OUTER FOLD 1/5
Train      : 367 samples / 62 diplotypes
Validation : 163 samples / 22 diplotypes
Outer Test : 92 samples / 20 diplotypes
Train class counts: {'de': 165, 'f': 133, 'nor': 69}
Val class counts  : {'nor': 85, 'de': 47, 'f': 31}
Test class counts : {'de': 38, 'f': 31, 'nor': 23}
Epoch [01/100] | Train Loss 1.0835 Acc 0.4278 | Val Loss 1.1312 Acc 0.3988 | Val Macro-Precision 0.4414 Macro-Recall 0.5269 Macro-F1 0.4081
Epoch [02/100] | Train Loss 1.0407 Acc 0.4905 | Val Loss 1.4649 Acc 0.4172 | Val Macro-Precision 0.4437 Macro-Recall 0.5591 Macro-F1 0.4350
Epoch [03/100] | Train Loss 1.0160 Acc 0.4877 | Val Loss 1.0866 Acc 0.4172 | Val Macro-Precision 0.4437 Macro-Recall 0.5591 Macro-F1 0.4350
Epoch [04/100] | Train Loss 0.9938 Acc 0.5259 | Val Loss 1.1110 Acc 0.4233 | Val Macro-Precision 0.4172 Macro-Recall 0.5772 Macro-F1 0.4437
Epoch [05/100] | Train Loss 0.9603 Acc 0.5940 | Val Loss 1.2042 Acc 0.4110 | Val Macro-Precision 0.4429 Macro-Recall 0.5484 Macro-F1 0.4264
Epoch [

,fold,best_epoch,n_train,n_val,n_test,n_train_groups,n_val_groups,n_test_groups,best_val_f1,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,nll,auroc_macro_ovr,aupr_macro_ovr
0,1,50,367,163,92,62,22,20,0.856781,0.500000,0.429487,0.470633,0.392747,0.423863,0.906710,0.795552,0.641357
1,2,57,475,67,80,70,18,16,0.674148,0.725000,0.444444,0.543126,0.467822,0.717278,0.620927,0.866858,0.679012
2,3,17,305,87,230,64,16,24,0.594547,0.273913,0.444056,0.305958,0.218747,0.208761,1.369863,0.568459,0.494264
3,4,55,341,149,132,61,20,23,0.871840,0.643939,0.641117,0.617858,0.617867,0.639009,0.948745,0.752986,0.645652
4,5,34,430,104,88,66,17,21,0.861808,0.522727,0.447856,0.334465,0.305408,0.446282,0.935548,0.710982,0.549757



MEAN ± SD ACROSS OUTER FOLDS


,mean,sd
accuracy,0.533116,0.171386
precision_macro,0.481392,0.089568
recall_macro,0.454408,0.133486
f1_macro,0.400518,0.153230
f1_weighted,0.487039,0.199497
auroc_macro_ovr,0.738968,0.111367
aupr_macro_ovr,0.602008,0.077009
nll,0.956359,0.267608



POOLED OUT-OF-FOLD PERFORMANCE
{'accuracy': 0.4790996784565916, 'precision_macro': 0.5193065706223601, 'recall_macro': 0.4580112994350283, 'f1_macro': 0.46489651101571683, 'f1_weighted': 0.4708974088640393, 'nll': 1.0542162022100454, 'auroc_macro_ovr': 0.721105183692098, 'aupr_macro_ovr': 0.5957302499175031}

Saved fold metrics, training histories, and OOF predictions.
